# Surround-grating analysis

This notebook analyzes `spotWithAnnularGrating` with the grating restricted to the
**surround**. It follows the same workflow as the cone center/annulus notebooks:
database refresh, condition discovery, one-cell analysis, persistent saving,
population analysis, and an example-stimulus visualization.

The light level is reconstructed per epoch block. Fixed `EL...` filters come from
the raw Stage device configurator; embedded `FW...` text there is ignored. The
numeric FilterWheel reading comes independently from protected metadata and must
agree across every epoch in a block. The resulting maximum is the calibrated R*/s
at normalized display intensity 1; the actual background is
`max_light_level × backgroundIntensity`.


In [ ]:
import sys
import time

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f'This notebook requires the retinanalysis Python 3.11 kernel; '
        f'got Python {sys.version.split()[0]} at {sys.executable}')

import_started = time.perf_counter()
import numpy as np
import pandas as pd
from IPython.display import display

import retinanalysis as ra
from retinanalysis.SCutils import explore as sc
from retinanalysis.SCutils.protocols import spot_annular_grating as sag

SITE = 'surround'
SITE_LABEL = 'Surround-grating analysis'
CELL_TYPES = ('ON-parasol',)
FILTER_WHEEL = (0.0, 0.5, 1.0)
BRIGHT_CONTRAST = (0.9, 1.0)
MIN_BAR_WIDTH = 40.0
MIN_EPOCHS = 15
MAX_SERIES_RESISTANCE = 30e6
STORE_PATH = sag.store_dir() / f'{SITE}_grating'
POPULATION_CONDITION = 'ON-parasol / surround'

print(f'Python {sys.version.split()[0]} | {sys.executable}')
print(f'Imports ready in {time.perf_counter() - import_started:.2f} s')
print(f'Saved records: {STORE_PATH}')


## 1. Load or refresh the single-cell database

Set `UPDATE_DATABASE = True` when new Symphony H5/JSON metadata should be ingested.
The default connects to the existing local DataJoint database without changing it.


In [ ]:
UPDATE_DATABASE = False
ra.djconnect()

if UPDATE_DATABASE:
    database_report = ra.populate_database()
    print(f"newly added: {len(database_report['added'])}")
    print(f"refreshed: {len(database_report['updated'])}")
    print(f"errored: {len(database_report['skipped'])}")
    print(f"database: {len(database_report['experiments'])} experiments; "
          f"{len(database_report['stale'])} still out of date")
else:
    print('Connected to the existing database. Set UPDATE_DATABASE=True to ingest changes.')


## 2. Search the database for surround-grating recordings

Discovery returns one row per epoch block, then validates `onlineAnalysis` against
the amplifier series resistance. A zero resistance supports cell-attached recording;
a positive resistance indicates whole-cell recording, and high-resistance epochs are
excluded by the existing pipeline. The table is then restricted to `surround` geometry
and grouped into explicit cell × mode × NDF combination × background conditions.

`ndf_combination`, `max_light_level`, and `rstar` are computed from the new block-level
light reader. No `maxIntensity` protocol parameter is assumed.


In [ ]:
df_blocks = sag.find_blocks(show=False)
df_blocks = sag.check_series_resistance(
    df_blocks, max_series_resistance=MAX_SERIES_RESISTANCE)
site_blocks = df_blocks[df_blocks.grating_site.eq(SITE)].copy()

groups = sag.group_blocks(
    site_blocks, show=False,
    allowed_filter_wheel=FILTER_WHEEL,
    allowed_bright_contrast=BRIGHT_CONTRAST,
    min_bar_width=MIN_BAR_WIDTH,
    min_epochs=MIN_EPOCHS)
selected = sag.select_cell_types(groups, cell_types=CELL_TYPES, show=False)
selected = selected.sort_values(
    ['exp_name', 'cell_label', 'onlineAnalysis', 'ndf_combination',
     'backgroundIntensity']).reset_index(drop=True)
selected.insert(0, 'date_index', pd.factorize(selected.exp_name, sort=False)[0] + 1)

print(f'{SITE_LABEL}: {len(selected)} conditions across '
      f'{selected.exp_name.nunique()} experiments and '
      f"{selected.groupby(['exp_name', 'cell_label']).ngroups} cells")
condition_columns = [
    'date_index', 'exp_name', 'cell_label', 'cell_type_short', 'onlineAnalysis',
    'ndf_combination', 'max_light_level', 'backgroundIntensity', 'rstar',
    'spot_intensity', 'bright', 'bar_width', 'aperture', 'annulus_inner',
    'annulus_outer', 'rs_mohm', 'blocks', 'epochs', 'block_ids',
]
condition_columns = [column for column in condition_columns if column in selected]
sc.scroll_table(
    selected[condition_columns], height=430,
    num_cols=('date_index', 'max_light_level', 'backgroundIntensity', 'rstar',
              'spot_intensity', 'aperture', 'annulus_inner', 'annulus_outer',
              'rs_mohm', 'blocks', 'epochs'))


## 3. Analyze one cell condition

Choose the condition explicitly by experiment, cell label, resolved recording mode,
combined NDF setting, and background intensity. This section can run after the import
cell without Section 2: it performs its own date-specific discovery and resistance check.

Before traces are loaded, the notebook prints the requested basic metadata: cell label
and type, block start times, recording mode and resistance, background and spot
intensities, bright-bar contrast, bar width, grating geometry, fixed filters, numeric
FilterWheel value, maximum light level, and actual background R*/s.


In [ ]:
EXP_NAME = '2026-05-27_G'
CELL_LABEL = 'Cell1'
ONLINE_ANALYSIS = 'extracellular'  # 'extracellular', 'exc', or 'inh'
NDF_COMBINATION = 'EL06 + EL2 + FW1'
BACKGROUND_INTENSITY = 0.15

date_blocks = sag.find_blocks(exp_names=[EXP_NAME], show=False)
date_blocks = sag.check_series_resistance(
    date_blocks, max_series_resistance=MAX_SERIES_RESISTANCE, show=False)
date_blocks = date_blocks[date_blocks.grating_site.eq(SITE)].copy()
date_groups = sag.group_blocks(
    date_blocks, show=False,
    allowed_filter_wheel=FILTER_WHEEL,
    allowed_bright_contrast=BRIGHT_CONTRAST,
    min_bar_width=MIN_BAR_WIDTH,
    min_epochs=MIN_EPOCHS)

condition_rows = date_groups[
    date_groups.cell_label.eq(CELL_LABEL)
    & date_groups.onlineAnalysis.eq(ONLINE_ANALYSIS)
    & date_groups.ndf_combination.eq(NDF_COMBINATION)
    & np.isclose(date_groups.backgroundIntensity, BACKGROUND_INTENSITY)
]
if len(condition_rows) != 1:
    available = date_groups[
        ['cell_label', 'cell_type_short', 'onlineAnalysis', 'ndf_combination',
         'backgroundIntensity', 'bright', 'bar_width', 'block_ids']]
    display(available)
    raise ValueError(f'Expected one matching condition, found {len(condition_rows)}')

condition_row = condition_rows.iloc[0]
condition_block_ids = [int(value) for value in condition_row.block_ids.split(',')]
condition_blocks = date_blocks[date_blocks.block_id.isin(condition_block_ids)].copy()
metadata = pd.DataFrame([{
    'cell_label': condition_row.cell_label,
    'cell_type': condition_row.cell_type_short,
    'block_start_times': ' | '.join(
        condition_blocks.start_time.astype(str).drop_duplicates()),
    'onlineAnalysis': condition_row.onlineAnalysis,
    'seriesResistance_MOhm': condition_row.get('rs_mohm', np.nan),
    'backgroundIntensity': condition_row.backgroundIntensity,
    'spotIntensity': condition_row.spot_intensity,
    'brightBarContrast': condition_row.bright,
    'barWidth_um': condition_row.bar_width,
    'apertureDiameter_um': condition_row.aperture,
    'annulusInnerDiameter_um': condition_row.annulus_inner,
    'annulusOuterDiameter_um': condition_row.annulus_outer,
    'fixed_NDFs_plus_FilterWheel': condition_row.ndf_combination,
    'max_light_level_Rstar_per_s': condition_row.max_light_level,
    'background_Rstar_per_s': condition_row.rstar,
    'block_ids': condition_row.block_ids,
    'epochs': condition_row.epochs,
}])
print('Selected cell metadata:')
display(metadata.T.rename(columns={0: 'value'}))

record = sag.analyze_group(
    EXP_NAME, condition_block_ids,
    online_analysis=ONLINE_ANALYSIS,
    max_series_resistance=MAX_SERIES_RESISTANCE,
    keep_raw=True)
sag.plot_group(record)


### 3a. Save this cell for population analysis

Save this exact condition to the site-specific HDF5 store. Re-running the same
cell/mode/site/NDF/background key replaces its record rather than duplicating it.
Center and surround notebooks use different folders, so pruning or population work
in one notebook cannot remove or mix records from the other.


In [ ]:
condition_output_path = sag.save_records([record], path=STORE_PATH)
print(condition_output_path)


### 3b. Check saved cells

Load only the scalar index. Fixed NDFs, numeric wheel, maximum light level,
background R*/s, recording mode, resistance, and block IDs remain visible without
loading trace arrays.


In [ ]:
saved_cells = sag.load_summary(path=STORE_PATH)
saved_columns = [
    'exp_name', 'cell_label', 'cell_type', 'online_analysis', 'ndf_combination',
    'max_light_level', 'background_intensity', 'rstar', 'series_resistance',
    'n_epochs_high_rs', 'n_epochs', 'block_ids',
]
saved_columns = [column for column in saved_columns if column in saved_cells]
print(f'{len(saved_cells)} saved {SITE}-grating condition(s)')
sc.scroll_table(
    saved_cells[saved_columns], height=320,
    num_cols=('max_light_level', 'background_intensity', 'rstar',
              'series_resistance', 'n_epochs_high_rs', 'n_epochs'))


## 4. Population analysis

Population work starts from the site-specific saved index. Recording modes remain
separate because firing rate and current have different units. The first figure shows
the measured cancellation against background light level and the Weber prediction;
the second shows the complete normalized tuning curves by light-level rung.


In [ ]:
summary = sag.add_condition(sag.load_summary(path=STORE_PATH))
if summary.empty:
    raise ValueError(f'No saved {SITE}-grating records in {STORE_PATH}')

population_table = (summary.groupby(
    ['cell_type', 'online_analysis', 'ndf_combination', 'rstar_level'],
    dropna=False)
    .agg(recordings=('key', 'size'), cells=('cell_label', 'nunique'),
         epochs=('n_epochs', 'sum'), mean_rstar=('rstar', 'mean'),
         crossing_mean=('crossing_interp', 'mean'),
         crossing_sem=('crossing_interp', 'sem'))
    .reset_index())
sc.scroll_table(
    population_table, height=320,
    num_cols=('rstar_level', 'recordings', 'cells', 'epochs', 'mean_rstar',
              'crossing_mean', 'crossing_sem'))

available_modes = tuple(summary.online_analysis.dropna().drop_duplicates())
sag.plot_weber_comparison(
    summary, conditions=(POPULATION_CONDITION,), modes=available_modes)
sag.plot_population_tuning(
    summary, conditions=(POPULATION_CONDITION,), modes=available_modes)


### 4a. Example saved recording from each mode

Show the most deeply sampled saved cell for the canonical population condition in
each available recording mode, including its baseline, measured crossing, and model
prediction.


In [ ]:
available_modes = tuple(summary.online_analysis.dropna().drop_duplicates())
sag.plot_condition_examples(
    sag.load_records(path=STORE_PATH),
    conditions=(POPULATION_CONDITION,),
    modes=available_modes)


## 5. Example surround-grating stimulus

Render the selected block using the recorded aperture, annulus, bar width,
background, spot intensity, and bright/dark contrasts. The geometry should visibly
place the grating over the **surround**; the center spot remains an independent protocol
parameter for surround recordings.


In [ ]:
example_block = int(condition_block_ids[0])
stim = ra.StimBlock(EXP_NAME, example_block, verbose=False)
example_parameters = stim.df_epochs['epoch_parameters'].iloc[0]
dark_values = np.sort(stim.df_epochs['currentDarkContrast'].dropna().unique())
example_dark = dark_values[[0, len(dark_values) // 2, -1]]
stimulus_figure = sag.plot_stimulus_schematic(
    example_parameters, dark_contrasts=example_dark)
